# Part 1: Data Assessment

## Objective

Examine the provided arXiv research papers, identify the available file types and metadata,
assess the quality of the data, and determine what information is usable for the RAG pipeline.

In [1]:
from pathlib import Path
import pandas as pd
from pypdf import PdfReader

cwd = Path.cwd()

if (cwd / "data").exists():
    data_dir = cwd / "data"
else:
    data_dir = cwd.parent / "data"

print("Data directory:", data_dir)

Data directory: /Users/fideltorresochoa/Desktop/BTT/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data


In [2]:
files = sorted([
    file for file in data_dir.iterdir()
    if file.is_file() and file.name != ".gitkeep"
])

print("number of data files:", len(files))

for file in files:
    print(file.name)

number of data files: 5
2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pdf


In [3]:
file_info = []

for file in files:
    file_info.append({
        "filename": file.name,
        "file_type": file.suffix.lower(),
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2)
    })

file_df = pd.DataFrame(file_info)
file_df

,filename,file_type,size_mb
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,.pdf,0.59
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,.pdf,1.48
2,2608.20319v1_Inducing Task Models from Compute...,.pdf,4.67
3,2608.20320v1_An Agentic Approach for Active Da...,.pdf,2.58
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,.pdf,3.63


In [4]:
paper_info = []

for file in files:
    if file.suffix.lower() == ".pdf":
        try:
            reader = PdfReader(file)
            metadata = reader.metadata

            pages_with_text = 0
            total_characters = 0

            for page in reader.pages:
                text = page.extract_text()

                if text and text.strip():
                    pages_with_text += 1
                    total_characters += len(text)

            paper_info.append({
                "filename": file.name,
                "title": metadata.get("/Title") if metadata else None,
                "author": metadata.get("/Author") if metadata else None,
                "num_pages": len(reader.pages),
                "pages_with_text": pages_with_text,
                "text_characters": total_characters,
                "readable": total_characters > 0
            })

        except Exception as e:
            paper_info.append({
                "filename": file.name,
                "title": None,
                "author": None,
                "num_pages": None,
                "pages_with_text": 0,
                "text_characters": 0,
                "readable": False
            })

papers_df = pd.DataFrame(paper_info)
papers_df

,filename,title,author,num_pages,pages_with_text,text_characters,readable
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,28,28,89597,True
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,19,19,76707,True
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,23,23,85873,True
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,25,25,73593,True
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,9,9,48596,True


In [5]:
papers_df.isnull().sum()

filename           0
title              0
author             0
num_pages          0
pages_with_text    0
text_characters    0
readable           0
dtype: int64

In [6]:
papers_df[[
    "filename",
    "title",
    "author",
    "num_pages",
    "pages_with_text",
    "text_characters",
    "readable"
]]

,filename,title,author,num_pages,pages_with_text,text_characters,readable
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,28,28,89597,True
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,19,19,76707,True
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,23,23,85873,True
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,25,25,73593,True
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,9,9,48596,True


In [7]:
for file in files:
    reader = PdfReader(file)
    metadata = reader.metadata

    print("\nFILE:", file.name)

    if metadata:
        print("Metadata fields:", list(metadata.keys()))
    else:
        print("No metadata found")


FILE: 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pd

In [8]:
reader = PdfReader(files[0])
sample_text = reader.pages[0].extract_text()

print(sample_text[:2000])

2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time
settings can improve quality and efficiency by routing queries to thespecialistwho can answer most
effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value
estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while ac-
curate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are
expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal
search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form
value-of-information expressions tha

In [9]:
for file in files:
    reader = PdfReader(file)
    text = reader.pages[0].extract_text()

    print("\nFILE:", file.name)
    print(text[:300])
    print("-" * 80)


FILE: 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architect
--------------------------------------------------------------------------------

FILE: 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
AI4AI-Bench: Benchmarking LLM Agents in
Algorithmic Design for Recursive Self-Improvement
Yizhe Chi †, Wenyi Li, Deyao Hong, Xiaoqiu Wang, Mingju Gao, Kaisen Yang, Bingxiang He, Y oujie Zheng, Calvin Xiao,
Qinhuai Na ‡
Navers Lab, Einsia.AI T singhua University
†Project Lead ‡Corresponding Author
Ab
--------------------------------------------------------------------------------

FILE: 2608.20319v1_Induci

In [10]:
print("duplicate filenames:", file_df["filename"].duplicated().sum())
print("duplicate titles:", papers_df["title"].duplicated().sum())

duplicate filenames: 0
duplicate titles: 0


In [11]:
metadata_rows = []

for file in files:
    reader = PdfReader(file)
    metadata = reader.metadata

    metadata_rows.append({
        "filename": file.name,
        "title": metadata.get("/Title"),
        "author": metadata.get("/Author"),
        "doi": metadata.get("/DOI"),
        "license": metadata.get("/License")
    })

metadata_df = pd.DataFrame(metadata_rows)
metadata_df

,filename,title,author,doi,license
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,https://doi.org/10.48550/arXiv.2608.20318,http://creativecommons.org/licenses/by/4.0/
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,https://doi.org/10.48550/arXiv.2608.20319,http://arxiv.org/licenses/nonexclusive-distrib...
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,https://doi.org/10.48550/arXiv.2608.20320,http://arxiv.org/licenses/nonexclusive-distrib...
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,https://doi.org/10.48550/arXiv.2608.20331,http://arxiv.org/licenses/nonexclusive-distrib...


## Data Quality Findings

- The dataset contains 5 unique arXiv research papers, all in PDF format.
- All 5 PDFs opened successfully and contained extractable text on every page.
- Titles and author information were available for all papers.
- DOI and license information were also available in the PDF metadata.
- No duplicate filenames or paper titles were found.
- No missing values were found in the fields collected during the assessment.
- The extracted text is readable overall, although some formatting artifacts are present, including missing spaces, merged words, and extra symbols around author names and affiliations.



### Usability for the RAG Pipeline

The following information should be useful for the RAG pipeline:
- Full paper text
- Paper titles
- Author information
- arXiv IDs from the filenames
- DOI and license information
- PDF page information that can be preserved later for citations and source tracking

The dataset looks usable for the RAG prototype. All 5 papers were readable, no duplicate filenames or titles were found, and the metadata needed for source tracking is available. Some formatting issues were found in the extracted text, including missing spaces, merged words, and extra symbols around author names and affiliations. These issues should be handled during the data cleaning and preprocessing stage before the text is split into chunks for retrieval.

Task #3 clean and procces data 

In [12]:
# extra imports , output dir

import re
import json
import unicodedata
from dataclasses import dataclass
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ingestion_pipeline")

processed_dir = data_dir.parent / "data_processed"
processed_dir.mkdir(exist_ok=True)

print("Processed output directory:", processed_dir)

Processed output directory: /Users/fideltorresochoa/Desktop/BTT/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed


In [13]:
# extract per page text

def extract_arxiv_id(filename: str) -> str:
    """Pull the arXiv ID (e.g. 2608.20316v1) off the front of the filename."""
    match = re.match(r"([\d]{4}\.[\d]{5}v\d+)", filename)
    return match.group(1) if match else filename


def extract_pdf_pages(file_path: Path) -> list[dict]:
    """Extract one record per page, with document-level metadata attached to every row."""
    reader = PdfReader(file_path)
    meta = reader.metadata or {}

    records = []
    for page_number, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        records.append({
            "arxiv_id": extract_arxiv_id(file_path.name),
            "filename": file_path.name,
            "title": meta.get("/Title"),
            "author": meta.get("/Author"),
            "doi": meta.get("/DOI"),
            "license": meta.get("/License"),
            "page_number": page_number,
            "num_pages": len(reader.pages),
            "raw_text": raw_text
        })
    return records


all_pages = []
for file in files:
    logger.info(f"Extracting {file.name}")
    all_pages.extend(extract_pdf_pages(file))

pages_df = pd.DataFrame(all_pages)
print(f"Extracted {len(pages_df)} pages from {pages_df['filename'].nunique()} papers")
pages_df.head()

2026-09-15 11:33:12,529 | INFO | Extracting 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2026-09-15 11:33:13,775 | INFO | Extracting 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
2026-09-15 11:33:14,697 | INFO | Extracting 2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
2026-09-15 11:33:15,489 | INFO | Extracting 2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
2026-09-15 11:33:16,196 | INFO | Extracting 2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pdf


Extracted 104 pages from 5 papers


,arxiv_id,filename,title,author,doi,license,page_number,num_pages,raw_text
0,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,1,28,2026-08-21\nPandora’s AI Model Routing Box: Ef...
1,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,2,28,Pandora’s AI Model Routing Box: Efficient Allo...
2,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,3,28,Pandora’s AI Model Routing Box: Efficient Allo...
3,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,4,28,Pandora’s AI Model Routing Box: Efficient Allo...
4,2608.20316v1,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/,5,28,Pandora’s AI Model Routing Box: Efficient Allo...


In [14]:
# clean text

def clean_text(text: str) -> str:
    """Clean raw PDF-extracted text based on artifacts identified in the Part 1 assessment."""
    if not text or not text.strip():
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"-\s*\n\s*", "", text)              # rejoin hyphenated line breaks
    text = re.sub(r"(\w)-\s+(\w)", r"\1-\2", text)      # rejoin hyphenated stray spacing
    text = re.sub(r"\s[\*†‡]\s", " ", text)             # strip isolated footnote markers
    text = re.sub(r"\n+", " ", text)                    # collapse newlines
    text = re.sub(r"[ \t]+", " ", text)                 # collapse repeated whitespace

    return text.strip()


pages_df["cleaned_text"] = pages_df["raw_text"].apply(clean_text)
pages_df["char_count"] = pages_df["cleaned_text"].str.len()
pages_df["word_count"] = pages_df["cleaned_text"].str.split().str.len()

print("BEFORE:\n", pages_df.loc[0, "raw_text"][:300])
print("\nAFTER:\n", pages_df.loc[0, "cleaned_text"][:300])

BEFORE:
 2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architect

AFTER:
 2026-08-21 Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate Larson, Jacob Eisenstein∗ Google DeepMind Heterogeneous AI systems composed of multiple models, architect


In [15]:
# chunck text

CHUNK_SIZE_WORDS = 250
CHUNK_OVERLAP_WORDS = 40


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE_WORDS, overlap: int = CHUNK_OVERLAP_WORDS) -> list[str]:
    """Split text into overlapping word-based chunks. Returns [] for empty text."""
    words = text.split()
    if not words:
        return []
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


chunk_records = []
for _, row in pages_df.iterrows():
    page_chunks = chunk_text(row["cleaned_text"])
    for chunk_index, chunk in enumerate(page_chunks):
        chunk_records.append({
            "chunk_id": f"{row['arxiv_id']}_p{row['page_number']}_c{chunk_index}",
            "arxiv_id": row["arxiv_id"],
            "filename": row["filename"],
            "title": row["title"],
            "author": row["author"],
            "doi": row["doi"],
            "license": row["license"],
            "page_number": row["page_number"],
            "chunk_index": chunk_index,
            "text": chunk,
            "word_count": len(chunk.split())
        })

chunks_df = pd.DataFrame(chunk_records)
print(f"Produced {len(chunks_df)} chunks from {pages_df['filename'].nunique()} papers")
chunks_df[["chunk_id", "page_number", "word_count"]].head(10)

Produced 300 chunks from 5 papers


,chunk_id,page_number,word_count
0,2608.20316v1_p1_c0,1,250
1,2608.20316v1_p1_c1,1,250
2,2608.20316v1_p1_c2,1,124
3,2608.20316v1_p2_c0,2,250
4,2608.20316v1_p2_c1,2,250
5,2608.20316v1_p2_c2,2,204
6,2608.20316v1_p3_c0,3,250
7,2608.20316v1_p3_c1,3,250
8,2608.20316v1_p3_c2,3,244
9,2608.20316v1_p3_c3,3,34


In [16]:
# validate

assert not chunks_df.empty, "No chunks were produced"
assert chunks_df["chunk_id"].is_unique, "Duplicate chunk IDs found"
assert (chunks_df["word_count"] > 0).all(), "Empty chunks found"
assert chunks_df["title"].notna().all(), "Chunks missing title metadata"
assert chunks_df["arxiv_id"].notna().all(), "Chunks missing arxiv_id"
assert (chunks_df["word_count"] <= CHUNK_SIZE_WORDS + 5).all(), "Chunk exceeds target size unexpectedly"

print("All validation checks passed.")
print("\nChunk word-count distribution:")
print(chunks_df["word_count"].describe())

All validation checks passed.

Chunk word-count distribution:
count    300.000000
mean     201.453333
std       73.333716
min        2.000000
25%      149.500000
50%      250.000000
75%      250.000000
max      250.000000
Name: word_count, dtype: float64


In [17]:
#save 
#install pyarrow

%pip install pyarrow

jsonl_path = processed_dir / "chunks.jsonl"
parquet_path = processed_dir / "chunks.parquet"

with open(jsonl_path, "w") as f:
    for record in chunks_df.to_dict(orient="records"):
        f.write(json.dumps(record) + "\n")

chunks_df.to_parquet(parquet_path, index=False)

logger.info(f"Saved {len(chunks_df)} chunks to {jsonl_path}")
logger.info(f"Saved {len(chunks_df)} chunks to {parquet_path}")

Note: you may need to restart the kernel to use updated packages.


2026-09-15 11:33:31,573 | INFO | Saved 300 chunks to /Users/fideltorresochoa/Desktop/BTT/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chunks.jsonl
2026-09-15 11:33:31,573 | INFO | Saved 300 chunks to /Users/fideltorresochoa/Desktop/BTT/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chunks.parquet


In [18]:
# wrap as a repeatable pipeline 

@dataclass
class IngestionConfig:
    data_dir: Path
    output_dir: Path
    chunk_size_words: int = CHUNK_SIZE_WORDS
    chunk_overlap_words: int = CHUNK_OVERLAP_WORDS


class ArxivIngestionPipeline:
    def __init__(self, config: IngestionConfig):
        self.config = config
        self.pages_df = None
        self.chunks_df = None

    def extract(self):
        pdf_files = sorted([f for f in self.config.data_dir.iterdir() if f.suffix.lower() == ".pdf"])
        records = []
        for file in pdf_files:
            logger.info(f"Extracting {file.name}")
            records.extend(extract_pdf_pages(file))
        self.pages_df = pd.DataFrame(records)
        return self

    def clean(self):
        self.pages_df["cleaned_text"] = self.pages_df["raw_text"].apply(clean_text)
        return self

    def chunk(self):
        records = []
        for _, row in self.pages_df.iterrows():
            page_chunks = chunk_text(
                row["cleaned_text"],
                chunk_size=self.config.chunk_size_words,
                overlap=self.config.chunk_overlap_words
            )
            for chunk_index, chunk in enumerate(page_chunks):
                records.append({
                    "chunk_id": f"{row['arxiv_id']}_p{row['page_number']}_c{chunk_index}",
                    "arxiv_id": row["arxiv_id"],
                    "filename": row["filename"],
                    "title": row["title"],
                    "author": row["author"],
                    "doi": row["doi"],
                    "license": row["license"],
                    "page_number": row["page_number"],
                    "chunk_index": chunk_index,
                    "text": chunk,
                    "word_count": len(chunk.split())
                })
        self.chunks_df = pd.DataFrame(records)
        return self

    def validate(self):
        assert not self.chunks_df.empty, "No chunks produced"
        assert self.chunks_df["chunk_id"].is_unique, "Duplicate chunk IDs"
        assert (self.chunks_df["word_count"] > 0).all(), "Empty chunks found"
        assert self.chunks_df["title"].notna().all(), "Missing title metadata"
        return self

    def load(self):
        self.config.output_dir.mkdir(exist_ok=True)
        jsonl_path = self.config.output_dir / "chunks.jsonl"
        with open(jsonl_path, "w") as f:
            for record in self.chunks_df.to_dict(orient="records"):
                f.write(json.dumps(record) + "\n")
        self.chunks_df.to_parquet(self.config.output_dir / "chunks.parquet", index=False)
        logger.info(f"Saved {len(self.chunks_df)} chunks to {self.config.output_dir}")
        return self

    def run(self):
        return self.extract().clean().chunk().validate().load()


config = IngestionConfig(data_dir=data_dir, output_dir=processed_dir)
pipeline = ArxivIngestionPipeline(config)
pipeline.run()
print(f"Pipeline complete: {len(pipeline.chunks_df)} chunks ready for embedding")

2026-09-15 11:34:16,871 | INFO | Extracting 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2026-09-15 11:34:18,177 | INFO | Extracting 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
2026-09-15 11:34:19,242 | INFO | Extracting 2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
2026-09-15 11:34:20,023 | INFO | Extracting 2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
2026-09-15 11:34:20,770 | INFO | Extracting 2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pdf
2026-09-15 11:34:21,408 | INFO | Saved 300 chunks to /Users/fideltorresochoa/Desktop/BTT/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed


Pipeline complete: 300 chunks ready for embedding


## Summary

- Extracted per-page text and metadata for all 5 papers.
- Cleaned hyphenation breaks, stray footnote symbols, and irregular whitespace found during Part 1.
- Chunked text at the page level (250 words, 40-word overlap) so every chunk keeps an accurate page number for citations.
- Validated that every chunk has complete metadata, a unique ID, and non-empty text.
- Packaged the flow into `ArxivIngestionPipeline` for repeatable reruns.
- Output: `data_processed/chunks.jsonl` and `chunks.parquet`, ready for embedding.